# Description

In [1]:
"""
Generate Mock CTSI-Format Data Across Multiple IRBs
====================================================
Produces Epic Clarity/Caboodle-style CSVs organized by IRB protocol,
with overlapping patients across IRBs and simple numeric imaging data.

Output structure:
  data/raw/
    irb_2025_001/
      ehr/  (CSVs: Patient_Demographics, Encounters, Labs, etc.)
      ct/   (JSON files with numeric arrays per patient)
      mri/  (empty for IRB 1)
    irb_2025_002/
      ehr/
      ct/   (empty for IRB 2)
      mri/  (JSON files with numeric arrays per patient)
    irb_2025_003/
      ehr/
      ct/
      mri/
    hidden/  (IRB 3 moved here for permission exercises)
"""


'\nGenerate Mock CTSI-Format Data Across Multiple IRBs\n====================================================\nProduces Epic Clarity/Caboodle-style CSVs organized by IRB protocol,\nwith overlapping patients across IRBs and simple numeric imaging data.\n\nOutput structure:\n  data/raw/\n    irb_2025_001/\n      ehr/  (CSVs: Patient_Demographics, Encounters, Labs, etc.)\n      ct/   (JSON files with numeric arrays per patient)\n      mri/  (empty for IRB 1)\n    irb_2025_002/\n      ehr/\n      ct/   (empty for IRB 2)\n      mri/  (JSON files with numeric arrays per patient)\n    irb_2025_003/\n      ehr/\n      ct/\n      mri/\n    hidden/  (IRB 3 moved here for permission exercises)\n'

In [2]:
# ── Imports ──
import os
import csv
import json
import random
from pathlib import Path
from datetime import datetime, timedelta


In [3]:
# ── Initialize ──
random.seed(42)
base = Path.cwd().parent / "data" / "raw"
print(f"Output directory: {base}\n")


Output directory: /app/data/raw



In [4]:
# ══════════════════════════════════════════════════════════════
# PATIENT PLAN
# ══════════════════════════════════════════════════════════════
#
# IRB 1: PAT-001 to PAT-010
#   EHR: all 10
#   CT:  PAT-001, PAT-003, PAT-004, PAT-006, PAT-008, PAT-009
#   MRI: none
#
# IRB 2: 4 overlap + 6 new (PAT-011 to PAT-016)
#   EHR: all 10
#   CT:  none
#   MRI: PAT-002, PAT-005, PAT-011 to PAT-015
#
# IRB 3: 6 overlap + 4 new (PAT-017 to PAT-020)
#   EHR: all 10
#   CT:  all 10
#   MRI: all 10

irb_plan = {
    "IRB-2025-001": {
        "patients": [f"PAT-{i:03d}" for i in range(1, 11)],
        "ehr": [f"PAT-{i:03d}" for i in range(1, 11)],
        "ct":  ["PAT-001", "PAT-003", "PAT-004", "PAT-006", "PAT-008", "PAT-009"],
        "mri": [],
        "start_date": datetime(2025, 3, 1),
    },
    "IRB-2025-002": {
        "patients": ["PAT-002", "PAT-005", "PAT-007", "PAT-010",
                      "PAT-011", "PAT-012", "PAT-013", "PAT-014", "PAT-015", "PAT-016"],
        "ehr": ["PAT-002", "PAT-005", "PAT-007", "PAT-010",
                "PAT-011", "PAT-012", "PAT-013", "PAT-014", "PAT-015", "PAT-016"],
        "ct":  [],
        "mri": ["PAT-002", "PAT-005", "PAT-011", "PAT-012", "PAT-013", "PAT-014", "PAT-015"],
        "start_date": datetime(2025, 6, 15),
    },
    "IRB-2025-003": {
        "patients": ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                      "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "ehr": ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "ct":  ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "mri": ["PAT-002", "PAT-005", "PAT-007", "PAT-010", "PAT-011", "PAT-016",
                "PAT-017", "PAT-018", "PAT-019", "PAT-020"],
        "start_date": datetime(2025, 10, 1),
    },
}


In [5]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════

def rand_date(start, spread_days=180):
    return start + timedelta(days=random.randint(0, spread_days))

def fmt_dt(dt):
    return dt.strftime("%m/%d/%Y %I:%M %p")

def fmt_d(dt):
    return dt.strftime("%m/%d/%Y")

def write_csv(path, headers, rows):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(headers)
        w.writerows(rows)
    return len(rows)

def abnormal_flag(val, ref_range):
    if not isinstance(val, (int, float)):
        return ""
    try:
        if "-" in ref_range:
            lo, hi = ref_range.split("-")
            if val < float(lo): return "L"
            if val > float(hi): return "H"
        elif ref_range.startswith("<"):
            if val >= float(ref_range[1:]): return "H"
    except (ValueError, TypeError):
        pass
    return ""


In [6]:
# ══════════════════════════════════════════════════════════════
# REFERENCE DATA (diagnosis, medication, lab pools)
# ══════════════════════════════════════════════════════════════

ild_dx = [
    ("J84.112", "Idiopathic pulmonary fibrosis"),
    ("J84.10",  "Pulmonary fibrosis, unspecified"),
    ("J84.89",  "Other specified interstitial pulmonary diseases"),
    ("J84.9",   "Interstitial pulmonary disease, unspecified"),
    ("J84.111", "Idiopathic interstitial pneumonia, NOS"),
    ("J84.2",   "Lymphoid interstitial pneumonia"),
    ("M34.81",  "Systemic sclerosis with lung involvement"),
    ("M05.10",  "RA with lung involvement"),
]

comorbidity_dx = [
    ("I10",    "Essential hypertension"),
    ("E11.9",  "Type 2 diabetes mellitus"),
    ("J44.1",  "COPD with acute exacerbation"),
    ("I48.91", "Atrial fibrillation"),
    ("N18.3",  "CKD stage 3"),
    ("E78.5",  "Dyslipidemia"),
    ("K21.0",  "GERD"),
    ("G47.33", "Obstructive sleep apnea"),
    ("F32.9",  "Major depressive disorder"),
    ("Z87.891","Hx nicotine dependence"),
    ("R06.00", "Dyspnea"),
    ("R05.9",  "Cough"),
]

med_list = [
    ("NINTEDANIB 150 MG CAPSULE",  "Nintedanib",  "Antifibrotic",     "150","mg","Oral","BID"),
    ("PIRFENIDONE 801 MG TABLET",  "Pirfenidone", "Antifibrotic",     "801","mg","Oral","TID"),
    ("PREDNISONE 20 MG TABLET",    "Prednisone",  "Corticosteroid",   "20","mg","Oral","Daily"),
    ("PREDNISONE 10 MG TABLET",    "Prednisone",  "Corticosteroid",   "10","mg","Oral","Daily"),
    ("MYCOPHENOLATE 500 MG TABLET","Mycophenolate","Immunosuppressant","500","mg","Oral","BID"),
    ("OMEPRAZOLE 20 MG CAPSULE",   "Omeprazole",  "PPI",              "20","mg","Oral","Daily"),
    ("FUROSEMIDE 40 MG TABLET",    "Furosemide",  "Loop Diuretic",    "40","mg","Oral","Daily"),
    ("LISINOPRIL 10 MG TABLET",    "Lisinopril",  "ACE Inhibitor",    "10","mg","Oral","Daily"),
    ("ATORVASTATIN 40 MG TABLET",  "Atorvastatin","Statin",           "40","mg","Oral","Daily"),
    ("ALBUTEROL HFA INHALER",      "Albuterol",   "Bronchodilator",   "2","puffs","Inhalation","Q4H PRN"),
    ("METFORMIN 1000 MG TABLET",   "Metformin",   "Antidiabetic",     "1000","mg","Oral","BID"),
    ("AMLODIPINE 5 MG TABLET",     "Amlodipine",  "Antihypertensive", "5","mg","Oral","Daily"),
    ("SERTRALINE 100 MG TABLET",   "Sertraline",  "Antidepressant",   "100","mg","Oral","Daily"),
    ("OXYGEN SUPPLEMENTAL",        "Oxygen",      "Respiratory",      "2-4","L/min","NC","Continuous"),
    ("APIXABAN 5 MG TABLET",       "Apixaban",    "Anticoagulant",    "5","mg","Oral","BID"),
]

lab_panels = [
    ("Sodium",        "2951-2", "mEq/L",   "136-145",  lambda: round(random.gauss(138, 5))),
    ("Potassium",     "2823-3", "mEq/L",   "3.5-5.0",  lambda: round(random.gauss(4.2, 0.5), 1)),
    ("Creatinine",    "2160-0", "mg/dL",   "0.7-1.3",  lambda: round(random.gauss(1.1, 0.4), 1)),
    ("BUN",           "3094-0", "mg/dL",   "7-20",     lambda: round(random.gauss(18, 8))),
    ("Hemoglobin",    "718-7",  "g/dL",    "12.0-17.5",lambda: round(random.gauss(13.0, 1.8), 1)),
    ("WBC",           "6690-2", "x10^9/L", "4.5-11.0", lambda: round(random.gauss(7.5, 2.5), 1)),
    ("Platelet Count","777-3",  "x10^9/L", "150-400",  lambda: round(random.gauss(250, 60))),
    ("NT-proBNP",     "33762-6","pg/mL",   "<125",     lambda: round(random.gauss(400, 300))),
    ("CRP",           "1988-5", "mg/L",    "<3.0",     lambda: round(random.gauss(5, 4), 1)),
    ("ANA",           "8061-4", "",         "Negative", lambda: random.choice(["Negative","Negative","Pos 1:80","Pos 1:160","Pos 1:320"])),
    ("RF",            "11572-5","IU/mL",   "<14",      lambda: round(random.gauss(12, 15))),
    ("HbA1c",         "4548-4", "%",       "<5.7",     lambda: round(random.gauss(6.5, 1.2), 1)),
    ("KL-6",          "",       "U/mL",    "<1000",    lambda: round(random.gauss(1200, 500))),
    ("SP-D",          "",       "ng/mL",   "<110",     lambda: round(random.gauss(120, 40))),
    ("LDH",           "2532-0", "U/L",     "140-280",  lambda: round(random.gauss(230, 60))),
    ("Albumin",       "1751-7", "g/dL",    "3.5-5.0",  lambda: round(random.gauss(3.8, 0.5), 1)),
]

providers = [
    "RAHAGHI, FARBOD", "KRISHNAMURTHY, RAVI", "SAGGAR, RAJAN",
    "LYNCH, JOSEPH", "BELPERIO, JOHN", "WEIGT, S. SAM", "SANTOS, KEVIN",
]


In [7]:
# ══════════════════════════════════════════════════════════════
# STABLE PATIENT REGISTRY
# ══════════════════════════════════════════════════════════════
# Build once so the same patient has consistent demographics
# across IRBs (as they would in a real EHR).

all_patient_ids = sorted(set(
    pid for plan in irb_plan.values() for pid in plan["patients"]
))

patient_registry = {}
for pid in all_patient_ids:
    patient_registry[pid] = {
        "pat_id":   pid,
        "mrn":      f"MRN{random.randint(1000000, 9999999)}",
        "dob":      datetime(random.randint(1942, 1985), random.randint(1,12), random.randint(1,28)),
        "sex":      random.choice(["Male", "Female"]),
        "race":     random.choice(["White","Black or African American","Asian",
                                    "Hispanic or Latino","Other","White","White","Asian"]),
        "ethnicity":random.choice(["Not Hispanic or Latino"]*4 + ["Hispanic or Latino","Unknown"]),
        "language": random.choice(["English"]*6 + ["Spanish","Vietnamese","Farsi"]),
        "marital":  random.choice(["Married","Single","Divorced","Widowed","Married","Married"]),
        "zip":      random.choice(["90024","90025","90034","90049","91101","91206","90045","90064"]),
        "bmi":      round(random.uniform(20, 38), 1),
        "smoking":  random.choice(["Former","Former","Former","Current","Current","Never"]),
        "ppd":      round(random.uniform(0.5, 2.5), 1),
        "yrs":      random.randint(10, 50),
        "quit_date":rand_date(datetime(2005,1,1), spread_days=6000),
        "primary_ild": random.choice(ild_dx),
        "comorbidities": random.sample(comorbidity_dx, random.randint(2, 5)),
        "chronic_meds":  random.sample(med_list, random.randint(3, 7)),
        "primary_provider": random.choice(providers),
        "death_date": None if random.random() > 0.1 else rand_date(datetime(2025,6,1), 200),
    }


In [8]:
# ══════════════════════════════════════════════════════════════
# EHR CSV GENERATOR
# ══════════════════════════════════════════════════════════════

def generate_ehr_csvs(ehr_dir, patient_ids, irb_id, start_date):
    """Generate all CTSI-format CSVs for a list of patients under one IRB."""

    # ── Patient_Identifiers ──
    pi_rows = [[patient_registry[pid]["pat_id"], patient_registry[pid]["mrn"], "MRN",
                patient_registry[pid]["mrn"]] for pid in patient_ids]
    write_csv(os.path.join(ehr_dir, "Patient_Identifiers.csv"),
              ["PAT_ID","PAT_MRN_ID","IDENTITY_TYPE","IDENTITY_ID"], pi_rows)

    # ── Patient_Demographics ──
    pd_rows = []
    for pid in patient_ids:
        p = patient_registry[pid]
        quit_d = "" if p["smoking"] in ("Current","Never") else fmt_d(p["quit_date"])
        ppd    = "" if p["smoking"] == "Never" else str(p["ppd"])
        yrs    = "" if p["smoking"] == "Never" else str(p["yrs"])
        death  = fmt_d(p["death_date"]) if p["death_date"] else ""
        pd_rows.append([p["pat_id"], fmt_d(p["dob"]), death, p["sex"], p["race"],
                        p["ethnicity"], p["language"], p["marital"], p["zip"], "", "CA",
                        "Los Angeles", p["bmi"], p["smoking"], ppd, yrs, quit_d])
    write_csv(os.path.join(ehr_dir, "Patient_Demographics.csv"),
              ["PAT_ID","BIRTH_DATE","DEATH_DATE","SEX","RACE","ETHNIC_GROUP",
               "PREFERRED_LANGUAGE","MARITAL_STATUS","ZIP","CITY","STATE","COUNTY",
               "BMI","SMOKING_STATUS","TOBACCO_PAK_PER_DY","TOBACCO_USED_YEARS",
               "SMOKING_QUIT_DATE"], pd_rows)

    # ── Encounters ──
    enc_rows = []
    enc_map = {}
    enc_types = ["Office Visit"]*3 + ["ED Visit","Inpatient","Telehealth","Procedure"]
    csn = random.randint(8000000, 8100000)
    for pid in patient_ids:
        p = patient_registry[pid]
        p_encs = []
        for _ in range(random.randint(3, 10)):
            csn += random.randint(1, 80)
            dt = rand_date(start_date, 300)
            etype = random.choice(enc_types)
            dept = "EMERGENCY MEDICINE" if etype == "ED Visit" else random.choice(
                ["PULMONARY MEDICINE"]*3 + ["INTERNAL MEDICINE","RHEUMATOLOGY","THORACIC SURGERY"])
            los = str(random.randint(2, 14)) if etype == "Inpatient" else ""
            disp = random.choice(["Home","Home Health","SNF","Home"]) if etype == "Inpatient" else ""
            adt_map = {"Office Visit":"Outpatient","Telehealth":"Outpatient","ED Visit":"Emergency",
                       "Inpatient":"Inpatient","Procedure":"Outpatient"}
            ed_disp = random.choice(["Discharge","Admit","Discharge"]) if etype == "ED Visit" else ""
            enc_rows.append([pid, str(csn), fmt_dt(dt), etype, dept, random.choice(providers),
                            los, disp, adt_map.get(etype,"Outpatient"), ed_disp, "",
                            p["primary_ild"][0], p["primary_ild"][1]])
            p_encs.append((str(csn), dt, etype))
        enc_map[pid] = p_encs
    write_csv(os.path.join(ehr_dir, "Encounters.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ENCOUNTER_DATE","ENCOUNTER_TYPE","DEPARTMENT_NAME",
               "VISIT_PROVIDER","LOS_DAYS","DISCHARGE_DISPOSITION","ADT_PAT_CLASS",
               "ED_DISPOSITION","HOSPITAL_AREA","PRIMARY_DX_ICD10","PRIMARY_DX_NAME"], enc_rows)

    # ── Encounter_Diagnoses ──
    all_dx = ild_dx + comorbidity_dx
    ed_rows = []
    for pid in patient_ids:
        for csn, dt, _ in enc_map[pid]:
            chosen = [patient_registry[pid]["primary_ild"]] + random.sample(all_dx, random.randint(1, 5))
            for line, dx in enumerate(chosen, 1):
                ed_rows.append([pid, csn, fmt_d(dt), dx[0], dx[1], line,
                               "Y" if line == 1 else "N", "Active"])
    write_csv(os.path.join(ehr_dir, "Encounter_Diagnoses.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","CONTACT_DATE","DX_ICD10_CODE","DX_NAME",
               "DX_LINE","PRIMARY_YN","CURRENT_STATUS"], ed_rows)

    # ── Problem_Lists ──
    pl_rows = []
    for pid in patient_ids:
        p = patient_registry[pid]
        probs = [p["primary_ild"]] + p["comorbidities"]
        for i, dx in enumerate(probs):
            entry = rand_date(start_date, 200)
            resolved = "" if random.random() > 0.15 else fmt_d(rand_date(entry, 100))
            pl_rows.append([pid, dx[0], dx[1], fmt_d(entry), resolved,
                           "Resolved" if resolved else "Active", fmt_d(entry),
                           "Y" if i == 0 else "N"])
    write_csv(os.path.join(ehr_dir, "Problem_Lists.csv"),
              ["PAT_ID","DX_ICD10_CODE","DX_NAME","DATE_OF_ENTRY","RESOLVED_DATE",
               "STATUS","NOTED_DATE","PRINCIPAL_PL_YN"], pl_rows)

    # ── Labs ──
    lab_rows = []
    order_id = random.randint(900000, 910000)
    for pid in patient_ids:
        for csn, dt, _ in enc_map[pid]:
            for name, loinc, unit, ref, gen_val in random.sample(lab_panels, random.randint(4, len(lab_panels))):
                order_id += 1
                val = gen_val()
                val_str = str(max(0, val)) if isinstance(val, (int, float)) else val
                result_dt = dt + timedelta(hours=random.randint(1, 36))
                lab_rows.append([pid, csn, str(order_id), fmt_dt(dt), fmt_dt(result_dt),
                                name, loinc, val_str, unit, ref, abnormal_flag(val, ref),
                                "Completed", "Final"])
    write_csv(os.path.join(ehr_dir, "Labs.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ORDER_ID","ORDER_DATE","RESULT_DATE",
               "COMPONENT_NAME","LOINC_CODE","ORD_VALUE","REFERENCE_UNIT",
               "REFERENCE_RANGE","ABNORMAL_FLAG","ORDER_STATUS","LAB_STATUS"], lab_rows)

    # ── Medications ──
    med_rows = []
    med_oid = random.randint(500000, 510000)
    for pid in patient_ids:
        for med in patient_registry[pid]["chronic_meds"]:
            med_oid += 1
            start = rand_date(datetime(2022, 1, 1), 900)
            end = "" if random.random() > 0.25 else fmt_d(rand_date(start, 200))
            status = "Discontinued" if end else "Active"
            discon = random.choice(["","Side Effects","Completed","Changed"]) if end else ""
            csn = random.choice(enc_map[pid])[0]
            med_rows.append([pid, csn, str(med_oid), med[0], med[1], med[2],
                            med[3], med[4], med[5], med[6], fmt_d(start), end,
                            status, discon, f"Take {med[3]} {med[4]} by {med[5]} {med[6]}"])
    write_csv(os.path.join(ehr_dir, "Medications.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ORDER_ID","MEDICATION_NAME","GENERIC_NAME",
               "THERAPEUTIC_CLASS","DOSE","DOSE_UNIT","ROUTE","FREQUENCY",
               "START_DATE","END_DATE","ORDER_STATUS","DISCON_REASON","SIG"], med_rows)

    # ── Flowsheet_Vitals ──
    vital_defs = [
        ("Blood Pressure Systolic", "mmHg",       lambda: round(random.gauss(130, 18))),
        ("Blood Pressure Diastolic","mmHg",       lambda: round(random.gauss(76, 12))),
        ("Heart Rate",              "bpm",        lambda: round(random.gauss(80, 15))),
        ("Respiratory Rate",        "breaths/min",lambda: round(random.gauss(18, 4))),
        ("Temperature",             "F",          lambda: round(random.gauss(98.4, 0.6), 1)),
        ("SpO2",                    "%",          lambda: min(100, round(random.gauss(94, 3)))),
        ("Weight",                  "kg",         lambda: round(random.gauss(78, 15), 1)),
        ("O2 Flow Rate",           "L/min",      lambda: random.choice([0,0,0,2,2,3,4,6])),
    ]
    vs_rows = []
    for pid in patient_ids:
        for csn, dt, etype in enc_map[pid]:
            readings = random.randint(6, 24) if etype == "Inpatient" else 1
            for r in range(readings):
                r_dt = dt + timedelta(hours=r * random.randint(4, 8)) if readings > 1 else dt
                for vname, vunit, vgen in vital_defs:
                    vs_rows.append([pid, csn, fmt_dt(r_dt), vname, str(vgen()), vunit, "", ""])
    write_csv(os.path.join(ehr_dir, "Flowsheet_Vitals.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","RECORDED_DATE","VITAL_SIGN_TYPE",
               "VITAL_VALUE","VITAL_UNIT","TEMPLATE","MEAS_COMMENT"], vs_rows)

    # ── Imaging (orders/metadata) ──
    img_procs = [
        ("CT CHEST WITHOUT CONTRAST HIGH RESOLUTION","CT","CHEST"),
        ("CT CHEST WITH CONTRAST","CT","CHEST"),
        ("XR CHEST 2 VIEW","XR","CHEST"),
        ("PFT SPIROMETRY WITH DLCO","PFT","LUNGS"),
        ("ECHOCARDIOGRAM TTE","US","HEART"),
        ("6 MINUTE WALK TEST","OTHER","LUNGS"),
    ]
    img_rows = []
    img_oid = random.randint(200000, 210000)
    for pid in patient_ids:
        for csn, dt, _ in enc_map[pid]:
            for _ in range(random.randint(0, 2)):
                img_oid += 1
                proc = random.choice(img_procs)
                img_rows.append([pid, csn, str(img_oid), fmt_dt(dt), proc[0], proc[1],
                                proc[2], "Completed", f"IMG{random.randint(100000,999999)}",
                                "RADIOLOGY"])
    write_csv(os.path.join(ehr_dir, "Imaging.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ORDER_ID","ORDER_DATE","PROCEDURE_NAME",
               "MODALITY","BODY_PART","ORDER_STATUS","ACCESSION_NUMBER","PERFORMING_DEPT"],
              img_rows)

    # ── Imaging_Impressions_DEID ──
    impressions = [
        "1. Bilateral basilar honeycombing with traction bronchiectasis. Definite UIP.",
        "1. Bilateral GGO with subpleural sparing. Probable NSIP.",
        "1. Diffuse GGO, lower lobe predominant. Favor DIP.",
        "1. Bilateral reticulation. Indeterminate for UIP - recommend MDD.",
        "1. Progressive fibrosis. Increased honeycombing vs prior.",
        "1. Centrilobular nodules, upper lobe GGO. Consider RB-ILD or HP.",
        "1. Stable bilateral basilar reticulation. No interval change.",
        "1. Near-complete resolution of GGO post steroid therapy.",
    ]
    imp_rows = [[r[0], r[1], r[2], r[8], random.choice(impressions)]
                for r in img_rows if "CT" in r[4]]
    write_csv(os.path.join(ehr_dir, "Imaging_Impressions_DEID.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ORDER_ID","ACCESSION_NUMBER",
               "IMPRESSION_TEXT_DEID"], imp_rows)

    # ── Imaging_Narratives_DEID ──
    narr_rows = [[r[0], r[1], r[2], r[8],
                  "EXAM: CT CHEST\nTECHNIQUE: Helical, 1.0mm.\n"
                  "FINDINGS: See impression.\nIMPRESSION: " + random.choice(impressions)]
                 for r in img_rows if "CT" in r[4]]
    write_csv(os.path.join(ehr_dir, "Imaging_Narratives_DEID.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ORDER_ID","ACCESSION_NUMBER",
               "NARRATIVE_TEXT_DEID"], narr_rows)

    # ── Procedures ──
    proc_pool = [
        ("BRONCHOSCOPY WITH TRANSBRONCHIAL BIOPSY","31628"),
        ("BRONCHOSCOPY WITH BAL","31624"),
        ("PFT COMPLETE","94010"),("DLCO","94729"),
        ("6 MINUTE WALK TEST","94618"),("RIGHT HEART CATH","93451"),
        ("VATS WEDGE BIOPSY","32608"),("ECHOCARDIOGRAM","93306"),
    ]
    proc_rows = []
    proc_oid = random.randint(300000, 310000)
    for pid in patient_ids:
        for _ in range(random.randint(1, 4)):
            proc_oid += 1
            proc = random.choice(proc_pool)
            csn, dt, _ = random.choice(enc_map[pid])
            proc_rows.append([pid, csn, str(proc_oid), fmt_d(dt), proc[0], proc[1],
                             random.choice(providers), "Completed"])
    write_csv(os.path.join(ehr_dir, "Procedures.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","ORDER_ID","PROCEDURE_DATE","PROCEDURE_NAME",
               "CPT_CODE","PERFORMING_PROVIDER","PROCEDURE_STATUS"], proc_rows)

    # ── Social_History ──
    sh_rows = []
    for pid in patient_ids:
        p = patient_registry[pid]
        for csn, dt, _ in random.sample(enc_map[pid], min(2, len(enc_map[pid]))):
            ppd_s = "" if p["smoking"] == "Never" else str(p["ppd"])
            yrs_s = "" if p["smoking"] == "Never" else str(p["yrs"])
            pky   = "" if p["smoking"] == "Never" else str(round(p["ppd"] * p["yrs"], 1))
            qd    = "" if p["smoking"] in ("Current","Never") else fmt_d(p["quit_date"])
            sh_rows.append([pid, fmt_d(dt), p["smoking"],
                           "" if p["smoking"]=="Never" else "Cigarettes",
                           ppd_s, yrs_s, pky, qd, p["smoking"],
                           random.choice(["None","Social","Moderate","None"]),
                           "", "", random.choice(["None"]*4+["Marijuana"]), "",
                           random.choice(["Retired","Teacher","Construction","Healthcare","Engineer"]),
                           random.choice(["High School","College","Bachelor","Master"])])
    write_csv(os.path.join(ehr_dir, "Social_History.csv"),
              ["PAT_ID","CONTACT_DATE","TOBACCO_USE","TOBACCO_TYPE","PACKS_PER_DAY",
               "YEARS_USED","PACK_YEARS","QUIT_DATE","SMOKING_STATUS","ALCOHOL_USE",
               "ALCOHOL_TYPE","DRINKS_PER_WEEK","DRUG_USE","DRUG_TYPE",
               "OCCUPATION","EDUCATION_LEVEL"], sh_rows)

    # ── Family_History ──
    fh_conds = [("Lung Cancer","C34.90"),("COPD","J44.9"),("CAD","I25.10"),
                ("Diabetes","E11.9"),("HTN","I10"),("Pulm Fibrosis","J84.10"),
                ("RA","M06.9"),("Stroke","I63.9")]
    rels = ["Mother","Father","Sister","Brother","Mat. Grandmother","Pat. Grandfather"]
    fh_rows = []
    for pid in patient_ids:
        for _ in range(random.randint(1, 4)):
            c = random.choice(fh_conds)
            fh_rows.append([pid, random.choice(rels), c[0], c[1],
                           str(random.randint(40,80)) if random.random()>0.3 else "",
                           random.choice(["Yes","No","Unknown"]), ""])
    write_csv(os.path.join(ehr_dir, "Family_History.csv"),
              ["PAT_ID","RELATION","CONDITION","ICD10_CODE","AGE_AT_ONSET",
               "DECEASED","COMMENTS"], fh_rows)

    # ── Hospital_Unit_Transfers ──
    hut_rows = []
    for pid in patient_ids:
        for csn, dt, etype in enc_map[pid]:
            if etype == "Inpatient":
                unit = random.choice(["4NE","5NW","7WEST","MICU"])
                hut_rows.append([pid, csn, fmt_dt(dt), "Admission", "ED", unit, "", ""])
                d_dt = dt + timedelta(days=random.randint(2, 12))
                hut_rows.append([pid, csn, fmt_dt(d_dt), "Discharge", unit, "Discharge", "", ""])
    write_csv(os.path.join(ehr_dir, "Hospital_Unit_Transfers.csv"),
              ["PAT_ID","PAT_ENC_CSN_ID","EVENT_DATE","EVENT_TYPE","FROM_UNIT",
               "TO_UNIT","FROM_BED","TO_BED"], hut_rows)


In [9]:
# ══════════════════════════════════════════════════════════════
# IMAGING DATA GENERATORS (simple numeric arrays as JSON)
# ══════════════════════════════════════════════════════════════

def generate_ct_data(ct_dir, patient_ids, irb_id, start_date):
    """CT data as JSON: list of slices, each slice = list of ints (0-4095)."""
    for pid in patient_ids:
        n_slices = random.randint(5, 10)
        slice_size = 64
        slices = [[random.randint(0, 4095) for _ in range(slice_size)]
                  for _ in range(n_slices)]

        data = {
            "patient_id": pid,
            "irb_protocol": irb_id,
            "study_date": rand_date(start_date, 180).strftime("%Y-%m-%d"),
            "modality": "CT",
            "description": "HRCT CHEST WITHOUT CONTRAST",
            "slice_thickness_mm": 1.25,
            "pixel_spacing_mm": [0.5, 0.5],
            "num_slices": n_slices,
            "slice_size": slice_size,
            "note": "Synthetic. Each inner list = 1 axial slice, values 0-4095.",
            "pixel_data": slices,
        }
        with open(os.path.join(ct_dir, f"{pid}_ct.json"), "w") as f:
            json.dump(data, f, indent=2)


def generate_mri_data(mri_dir, patient_ids, irb_id, start_date):
    """MRI data as JSON: dict of sequences, each = list of slices, each slice = list of floats (0-1)."""
    all_seqs = ["T1", "T2", "STIR", "DWI"]
    for pid in patient_ids:
        n_slices = random.randint(3, 8)
        slice_size = 48
        seq_names = random.sample(all_seqs, random.randint(2, 4))
        seq_data = {seq: [[round(random.random(), 4) for _ in range(slice_size)]
                          for _ in range(n_slices)]
                    for seq in seq_names}

        data = {
            "patient_id": pid,
            "irb_protocol": irb_id,
            "study_date": rand_date(start_date, 180).strftime("%Y-%m-%d"),
            "modality": "MRI",
            "description": "CHEST MRI",
            "field_strength_T": random.choice([1.5, 3.0]),
            "sequences": seq_names,
            "num_slices_per_seq": n_slices,
            "slice_size": slice_size,
            "note": "Synthetic. Each sequence key maps to list of slices, values 0-1.",
            "pixel_data": seq_data,
        }
        with open(os.path.join(mri_dir, f"{pid}_mri.json"), "w") as f:
            json.dump(data, f, indent=2)


In [10]:
# ══════════════════════════════════════════════════════════════
# GENERATE ALL FILES
# ══════════════════════════════════════════════════════════════

file_count = 0

for irb_id, plan in irb_plan.items():
    irb_dir = base / irb_id.lower().replace("-", "_")

    ehr_dir = irb_dir / "ehr"
    ct_dir  = irb_dir / "ct"
    mri_dir = irb_dir / "mri"
    for d in [ehr_dir, ct_dir, mri_dir]:
        d.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Generating {irb_id}")
    print(f"{'='*60}")

    if plan["ehr"]:
        print(f"  EHR: {len(plan['ehr'])} patients -> 15 CSVs")
        generate_ehr_csvs(str(ehr_dir), plan["ehr"], irb_id, plan["start_date"])
        file_count += 15

    if plan["ct"]:
        print(f"  CT:  {len(plan['ct'])} patients -> {len(plan['ct'])} JSON files")
        generate_ct_data(str(ct_dir), plan["ct"], irb_id, plan["start_date"])
        file_count += len(plan["ct"])

    if plan["mri"]:
        print(f"  MRI: {len(plan['mri'])} patients -> {len(plan['mri'])} JSON files")
        generate_mri_data(str(mri_dir), plan["mri"], irb_id, plan["start_date"])
        file_count += len(plan["mri"])



Generating IRB-2025-001
  EHR: 10 patients -> 15 CSVs
  CT:  6 patients -> 6 JSON files

Generating IRB-2025-002
  EHR: 10 patients -> 15 CSVs
  MRI: 7 patients -> 7 JSON files

Generating IRB-2025-003
  EHR: 10 patients -> 15 CSVs
  CT:  10 patients -> 10 JSON files
  MRI: 10 patients -> 10 JSON files


In [11]:
# ══════════════════════════════════════════════════════════════
# HIDE IRB 3 (for permission exercises)
# ══════════════════════════════════════════════════════════════

hidden_dir = base / "hidden"
hidden_dir.mkdir(exist_ok=True)

src = base / "irb_2025_003"
dst = hidden_dir / "irb_2025_003"
if src.exists() and not dst.exists():
    src.rename(dst)
    print(f"\nMoved {src} -> {dst}")



Moved /app/data/raw/irb_2025_003 -> /app/data/raw/hidden/irb_2025_003


In [12]:
# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print("MOCK DATA SUMMARY")
print(f"{'='*60}")
print(f"\nTotal files generated: {file_count}")
print(f"Output directory: {base}\n")

for irb_id, plan in irb_plan.items():
    print(f"{irb_id}")
    print(f"  Patients: {', '.join(plan['patients'])}")
    print(f"  EHR: {len(plan['ehr'])} patients -> 15 CSVs")
    print(f"  CT:  {len(plan['ct'])} patients -> {len(plan['ct'])} JSON files")
    print(f"  MRI: {len(plan['mri'])} patients -> {len(plan['mri'])} JSON files")
    print()

irb1 = set(irb_plan["IRB-2025-001"]["patients"])
irb2 = set(irb_plan["IRB-2025-002"]["patients"])
irb3 = set(irb_plan["IRB-2025-003"]["patients"])

print("PATIENT OVERLAP")
print(f"  IRB 1 & IRB 2: {sorted(irb1 & irb2)}")
print(f"  IRB 1 & IRB 3: {sorted(irb1 & irb3)}")
print(f"  IRB 2 & IRB 3: {sorted(irb2 & irb3)}")
print(f"  All three:     {sorted(irb1 & irb2 & irb3)}")
print(f"\n  Unique patients: {len(irb1 | irb2 | irb3)}")

print(f"\nDIRECTORY TREE:")
for dirpath, dirnames, filenames in os.walk(base):
    depth = dirpath.replace(str(base), "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(dirpath)}/")
    if depth < 3:
        sub = "  " * (depth + 1)
        for f in sorted(filenames)[:6]:
            print(f"{sub}{f}")
        if len(filenames) > 6:
            print(f"{sub}... and {len(filenames) - 6} more")



MOCK DATA SUMMARY

Total files generated: 78
Output directory: /app/data/raw

IRB-2025-001
  Patients: PAT-001, PAT-002, PAT-003, PAT-004, PAT-005, PAT-006, PAT-007, PAT-008, PAT-009, PAT-010
  EHR: 10 patients -> 15 CSVs
  CT:  6 patients -> 6 JSON files
  MRI: 0 patients -> 0 JSON files

IRB-2025-002
  Patients: PAT-002, PAT-005, PAT-007, PAT-010, PAT-011, PAT-012, PAT-013, PAT-014, PAT-015, PAT-016
  EHR: 10 patients -> 15 CSVs
  CT:  0 patients -> 0 JSON files
  MRI: 7 patients -> 7 JSON files

IRB-2025-003
  Patients: PAT-002, PAT-005, PAT-007, PAT-010, PAT-011, PAT-016, PAT-017, PAT-018, PAT-019, PAT-020
  EHR: 10 patients -> 15 CSVs
  CT:  10 patients -> 10 JSON files
  MRI: 10 patients -> 10 JSON files

PATIENT OVERLAP
  IRB 1 & IRB 2: ['PAT-002', 'PAT-005', 'PAT-007', 'PAT-010']
  IRB 1 & IRB 3: ['PAT-002', 'PAT-005', 'PAT-007', 'PAT-010']
  IRB 2 & IRB 3: ['PAT-002', 'PAT-005', 'PAT-007', 'PAT-010', 'PAT-011', 'PAT-016']
  All three:     ['PAT-002', 'PAT-005', 'PAT-007', 'PA